In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq()
model = "llama-3.3-70b-versatile"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.chat.completions.create(**params)
    return message.choices[0].message.content

In [12]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,'```json')
    return chat(messages,stop_sequences=['```'])


In [18]:
dataset = json.loads(generate_dataset())
print(dataset)


[{'task': "Write a Python function to extract the 'Region' from an AWS ARN (Amazon Resource Name) given as a string. For example, given the ARN 'arn:aws:ec2:us-east-1:123456789012:instance/i-12345678', the function should return 'us-east-1'."}, {'task': 'Create a JSON object representing an AWS IAM policy that grants access to list all S3 buckets.'}, {'task': "Write a regular expression to match and extract the 'BucketName' from an AWS S3 URL, such as 'https://s3.amazonaws.com/my-bucket-name/path/to/object'. The regular expression should return 'my-bucket-name'."}]


In [20]:
with open('dataset.json','w') as f:
    json.dump(dataset,f,indent=2)

In [21]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [22]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [23]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [24]:
results = run_eval(dataset)

In [28]:
print(json.dumps(results,indent=2))

[
  {
    "output": "Here is a Python function to extract the region from an AWS ARN:\n\n```python\ndef extract_region_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the region from an AWS ARN.\n\n    Args:\n    arn (str): The AWS ARN from which to extract the region.\n\n    Returns:\n    str: The extracted region.\n    \"\"\"\n    # Split the ARN into its parts\n    arn_parts = arn.split(\":\")\n\n    # The region is the fourth part of the ARN (index 3)\n    region = arn_parts[3]\n\n    return region\n\n# Example usage:\narn = \"arn:aws:ec2:us-east-1:123456789012:instance/i-12345678\"\nprint(extract_region_from_arn(arn))  # Output: us-east-1\n```\n\nThis function works by splitting the ARN into its parts (using the colon as a delimiter) and then returning the fourth part, which corresponds to the region.",
    "test_case": {
      "task": "Write a Python function to extract the 'Region' from an AWS ARN (Amazon Resource Name) given as a string. For example, given the ARN 'arn:aws